# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [16]:
# constants

MODEL_GPT = 'gpt-6-luna'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [17]:
# imports and environment setup

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from enum import Enum
from openai import OpenAI

# Initialize and constants
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-6-luna'
# openai = OpenAI()

API key looks good so far


In [18]:
# setup client and call to client

class Provider(Enum):
    OPENAI = "OpenAI GPT 6 Luna "
    OLLAMA = "Llama 3.2"

def create_client(provider: Provider) -> OpenAI:
    match provider:
        case Provider.OPENAI:
            return OpenAI()
        case Provider.OLLAMA:
            return OpenAI(
                base_url=OLLAMA_BASE_URL,
                api_key="ollama"
            )
        case _:
            raise ValueError(f"Unsupported provider: {provider}")

def call_client(provider: Provider, system_prompt, user_prompt):
    client = create_client(provider)
    messages = [
                    { "role" : "system", "content" : system_prompt },
                    { "role" : "user", "content" : user_prompt }
                ]
    match provider:
        case Provider.OPENAI:
            return client.chat.completions.create(
                model=MODEL_GPT,
                reasoning_effort="none",
                messages=messages,
                stream=True
            )
        case Provider.OLLAMA:
            return client.chat.completions.create(
                model=MODEL_LLAMA,
                messages=messages,
                stream=True
            )
        case _:
            raise ValueError(f"Unsupported provider: {provider}")
            

In [ ]:
# setup system and user prompts, call client and stream response

def explain(concept, source, target, provider:Provider=Provider.OPENAI):
    system_prompt = """
        You are a programming tutor who is helping an experienced coder learn a new target
        language using their knowledge of a source language.
        
        Explain the requested concept in the target language, relating it to familar concepts
        in the source language where useful.

        Provide a brief explanation and 1-2 short target language code examples.
        Highlight important differences. If there is no direct equivalent, say so instead of forcing 
        an analogy.

        Assume general programming knowledge and focus on language-specific details.
    """

    user_prompt = f"""
        I want to understand how {concept} works in {target}. I am proficient and 
        have relevant work experience using {source} at a professional level. I'm trying to
        learn {target} using my understanding of {source}.
    """

    stream = call_client(
                provider=provider,
                system_prompt=system_prompt,
                user_prompt=user_prompt
            )

    response = f"### Model: {provider.value.upper()}\n\n"
    output = display(Markdown(""), display_id=True)


    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        output.update(Markdown(response))
    return response

In [28]:
gpt_answer = explain(
    concept="switch cases",
    source="Swift",
    target="Python",
    provider=Provider.OPENAI,
)

ollama_answer = explain(
    concept="switch cases",
    source="Swift",
    target="Python",
    provider=Provider.OLLAMA,
)

## Model: OPENAI GPT 6 LUNA 

Python doesn’t have a traditional `switch` statement like Swift’s. The closest built-in equivalent is **`match` / `case`**, introduced in Python 3.10.

```python
def describe(value):
    match value:
        case 0:
            return "zero"
        case 1 | 2:
            return "one or two"
        case _:
            return "something else"
```

`case _` is the catch-all, similar to Swift’s `default`. Python cases don’t fall through, so there’s no `break` to add. The `|` pattern matches either value.

`match` does more than compare a value against constants: it supports structural pattern matching, including unpacking sequences and matching object shapes.

```python
match point:
    case (0, 0):
        print("origin")
    case (x, y):
        print(f"coordinates: {x}, {y}")
```

Here `(x, y)` matches a two-item sequence and binds its items to variables. Be careful with bare names: `case name:` **binds** the matched value to `name`; it does not compare against an existing variable named `name`. Use a qualified constant such as `case Colors.red:` for a constant-value match.

For simple dispatch, Python also commonly uses `if`/`elif` or a dictionary of functions; `match` is most useful when matching several shapes or patterns.

## Model: LLAMA 3.2

As a Swift developer, you're already familiar with the concept of switch cases, also known as switch statements or switch expressions. In Python, the equivalent concept is the `match` statement.

**Python's `match` statement**

The `match` statement in Python is similar to the switch statement in Swift. It allows you to execute different blocks of code based on the value of a variable or expression. The main difference is that Python's `match` statement is more flexible and expressive.

Here's a brief overview of the syntax:

```python
match expression:
    case value1:
        # code to execute when expression equals value1
    case value2:
        # code to execute when expression equals value2
    case _:
        # code to execute when expression doesn't match any value
```

**Using `match` with integers**

Let's say you have an integer variable `x` and you want to execute different blocks of code based on its value. You can use the `match` statement like this:

```python
x = 5

match x:
    case 1:
        print("x is 1")
    case 2:
        print("x is 2")
    case 3:
        print("x is 3")
    case _:
        print("x is not 1, 2, or 3")
```

This will output: "x is 3"

**Using `match` with complex types**

Python's `match` statement can also be used with more complex types, such as lists, tuples, and dictionaries. For example:

```python
x = (1, 2, 3)

match x:
    case (1, 2, 3):
        print("x is (1, 2, 3)")
    case (4, 5, 6):
        print("x is (4, 5, 6)")
    case _:
        print("x is not (1, 2, 3) or (4, 5, 6)")
```

This will output: "x is (1, 2, 3)"

**Important differences**

While the `match` statement in Python is similar to the switch statement in Swift, there are some key differences:

* In Python, you don't need to specify a `default` branch like you do in Swift.
* Python's `match` statement can be used with more complex types, such as lists and dictionaries.
* Python's `match` statement is more expressive, allowing you to use patterns to match values, whereas Swift's switch statement requires a `case` label.

Overall, Python's `match` statement is a powerful tool that allows you to write more concise and expressive code. As you continue to learn Python, I encourage you to explore its various features and use cases!